# GPU training on Colab (T4): Methods 4, 5A, 5B

Trains every network in the project that needs a GPU, on a free Colab **T4**:

| Section | Method | What it trains | Rough T4 time |
|---|---|---|---|
| 4a | Method 4 (CayleyPy) | 4M-parameter ResMLP for the 3×3×3, 4×4×4, 5×5×5 (paper settings) | hours per cube (see the probe in §3) |
| 4b | Method 4 | per-N Square models, N = 3–8 (1M preset) | ~10 min each |
| 5A | Method 5A | cluster-token transformer: Squares (go / no-go) and cubes | ~1 h (Square), several h (cube) |
| 5B | Method 5B | per-cluster MLPs, larger than the committed CPU ones | ~1 h |

**How to use**
1. *Runtime → Change runtime type → T4 GPU*.
2. Edit the settings cell (§1), then *Runtime → Run all*.
3. Everything is written to Google Drive (`MyDrive/rubik_gpu/`). Colab disconnects after a few hours;
   just **run all again**: every training resumes from its last saved state, and finished models are skipped.
4. At the end (§7) a zip of all models and results is written to Drive. Copy the files into the repo's
   `checkpoints/` folders (names are given there) to use them in the solvers and the paper.

Times above are estimates; §3 measures your actual T4 speed before the long runs.

## 1. Settings

In [ ]:
# ---- what to run (set False to skip a section) ----
RUN_M4_CUBE   = True    # §4a  Method 4 cube models (long)
RUN_M4_SQUARE = True    # §4b  Method 4 Square models N = 3..8
RUN_5A_SQUARE = True    # §5a  Method 5A, Squares 4-8 -> tested on 10, 14, 20
RUN_5A_CUBE   = False   # §5b  Method 5A, cubes 3-7 (long)
RUN_5B        = True    # §6   Method 5B cluster MLPs
RUN_EVAL      = True    # evaluation cells after each section

# ---- Method 4 (paper: 8192 epochs x 1M examples = 8B; reduce to fit your time) ----
M4_CUBE_NS     = [3, 4, 5]
M4_EPOCHS      = 1024       # per cube size; the probe in §3 tells you how long this takes
M4_EPOCH_SIZE  = 1_000_000  # fresh examples per epoch (paper: 1M)
M4_BATCH       = 8192
M4_AMP         = True       # FP16 autocast: ~2-3x faster on a T4 (the paper trains in FP32)
M4_SQ_EPOCHS   = 256
M4_WIDTHS      = [1024, 4096, 16384, 65536, 262144]   # length-vs-W evaluation (paper: up to 2^18 / 2^24)

# ---- Method 5A ----
M5A_SQ_STEPS   = 20000
M5A_CUBE_STEPS = 50000

# ---- Method 5B ----
M5B_STEPS      = 20000
M5B_HIDDEN     = 512

# ---- repository ----
REPO   = "gauravrai1704/Rubik"
BRANCH = "full_proj"

## 2. GPU, Drive, code

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
OUT = "/content/drive/MyDrive/rubik_gpu"
for d in ("m4", "m5a", "m5b", "results", "logs"):
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
print("outputs ->", OUT)

The repository is cloned from GitHub. If it is **private**, add a Colab secret named `GITHUB_TOKEN`
(key icon in the left sidebar) holding a GitHub token with read access, and allow this notebook to use it.

In [ ]:
import os, subprocess
tok = ""
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass  # no secret: clone as a public repository
url = f"https://{tok}@github.com/{REPO}.git" if tok else f"https://github.com/{REPO}.git"
if not os.path.exists("/content/Rubik/.git"):
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/Rubik"],
                       capture_output=True, text=True)
    msg = (r.stdout + r.stderr)[-800:]
    print(msg.replace(tok, "***") if tok else msg)
    assert r.returncode == 0, "clone failed: private repo? add the GITHUB_TOKEN secret (see above)"
else:
    print(subprocess.run(["git", "-C", "/content/Rubik", "pull", "--ff-only"], capture_output=True, text=True).stdout)
%cd /content/Rubik
!git log --oneline -1

In [ ]:
# Colab already has PyTorch with CUDA; this adds the project and its other dependencies
!pip install -q -e . 2>&1 | tail -2
import rubiks, numba, kociemba
print("ok")

## 3. Speed probe
Two short Method 4 epochs on the 3×3×3 paper model, to measure examples per second on this GPU and
estimate the long runs before starting them.

In [ ]:
import time
from rubiks.solvers.m4_cayleypy.train import train
t = time.time()
_, meta = train("cube", 3, preset="paper3", epochs=2, epoch_size=M4_EPOCH_SIZE, batch=M4_BATCH,
                device="cuda", amp=M4_AMP, verbose=False)
rate = meta["examples"] / meta["train_time_s"]
print(f"{rate:,.0f} examples/s, peak GPU memory {meta['peak_gpu_mb']:.0f} MB")
for N, preset in ((3, "paper3"), (4, "paper4"), (5, "paper5")):
    if N in M4_CUBE_NS:
        hours = M4_EPOCHS * M4_EPOCH_SIZE / rate / 3600 * (1 if N == 3 else 1.6 if N == 4 else 2.4)
        print(f"cube {N}: {M4_EPOCHS} epochs ~ {hours:.1f} h (rough; bigger cubes have bigger inputs)")

## 4a. Method 4: cube models (paper settings)
One 4M-parameter, 10-layer ResMLP per size (Table II of the paper: `paper3`, `paper4`, `paper5`), walks of
length 26 / 45 / 65, MSE, Adam 1e-3. A training state is saved to Drive every 8 epochs; **rerunning this
cell resumes**. The finished model is `MyDrive/rubik_gpu/m4/cube<N>.large.pt` (+ `.json` with training
time and peak GPU memory).

In [ ]:
import os
if RUN_M4_CUBE:
    for N in M4_CUBE_NS:
        out = f"{OUT}/m4/cube{N}.large.pt"
        if os.path.exists(out):
            print(f"cube {N}: done ({out})"); continue
        !python -m rubiks.solvers.m4_cayleypy.train --puzzle cube --N {N} --preset paper{N} \
            --epochs {M4_EPOCHS} --epoch-size {M4_EPOCH_SIZE} --batch {M4_BATCH} --device cuda \
            {"--amp" if M4_AMP else ""} --save-every 8 --log-every 8 --out {out} 2>&1 | tee -a {OUT}/logs/m4_cube{N}.log

**Evaluate** (length vs beam width on random cubes; the paper's 3×3×3 target is 21.14 QTM at W = 2¹⁸).
Results go to `MyDrive/rubik_gpu/results/`.

In [ ]:
import os, shutil
if RUN_EVAL and RUN_M4_CUBE:
    os.makedirs("checkpoints/m4", exist_ok=True)
    ns = [N for N in M4_CUBE_NS if os.path.exists(f"{OUT}/m4/cube{N}.large.pt")]
    for N in ns:
        shutil.copy(f"{OUT}/m4/cube{N}.large.pt", f"checkpoints/m4/cube{N}.large.pt")
    if ns:
        !python -m rubiks.bench make --puzzle cube --N {",".join(map(str, ns))} --depth random --count 20
    for N in ns:
        for W in M4_WIDTHS:
            res = f"{OUT}/results/m4_cube{N}_W{W}.jsonl"
            if os.path.exists(res):
                continue
            !python -m rubiks.bench run --solver m4_cayleypy --puzzle cube --N {N} --dataset random \
                --device cuda --time 1200 --kw '{{"width": {W}}}' --out {res} | tail -3

## 4b. Method 4: Square models, N = 3..8

In [ ]:
if RUN_M4_SQUARE:
    for N in range(3, 9):
        out = f"{OUT}/m4/square{N}.large.pt"
        if os.path.exists(out):
            print(f"square {N}: done"); continue
        !python -m rubiks.solvers.m4_cayleypy.train --puzzle square --N {N} --preset 1M \
            --epochs {M4_SQ_EPOCHS} --epoch-size 1000000 --batch 8192 --device cuda {"--amp" if M4_AMP else ""} \
            --save-every 16 --log-every 16 --out {out} 2>&1 | tee -a {OUT}/logs/m4_square{N}.log | tail -3

In [ ]:
if RUN_EVAL and RUN_M4_SQUARE:
    for N in range(3, 9):
        if os.path.exists(f"{OUT}/m4/square{N}.large.pt"):
            shutil.copy(f"{OUT}/m4/square{N}.large.pt", f"checkpoints/m4/square{N}.large.pt")
    res = f"{OUT}/results/m4_square_W4096.jsonl"
    if not os.path.exists(res):
        !python -m rubiks.bench run --solver m4_cayleypy --puzzle square --N 3-8 --dataset random \
            --device cuda --time 600 --kw '{{"width": 4096}}' --out {res} | tail -8

## 5a. Method 5A: one network for every Square size (go / no-go, technical basis §8.6)
Trained on Squares N = 4..8 only, checked on N = 10, 14, 20 which it never sees.
State saved every 500 steps; rerunning resumes.

In [ ]:
if RUN_5A_SQUARE:
    out = f"{OUT}/m5a/square.large.pt"
    if not os.path.exists(out):
        !python -m rubiks.solvers.m5a_tokens.train --puzzle square --train-N 4-8 --val-N 10,14,20 \
            --steps {M5A_SQ_STEPS} --d-model 192 --layers 4 --heads 6 --device cuda \
            --save-every 500 --out {out} 2>&1 | tee -a {OUT}/logs/m5a_square.log | grep -E "step|resumed|saved"

In [ ]:
if RUN_EVAL and RUN_5A_SQUARE and os.path.exists(f"{OUT}/m5a/square.large.pt"):
    res = f"{OUT}/results/m5a_square_large.jsonl"
    if not os.path.exists(res):
        ck = f"{OUT}/m5a/square.large.pt"
        !python -m rubiks.bench run --solver m5a_tokens --puzzle square --N 4-10,12,14 --dataset random \
            --limit 5 --device cuda --time 900 --kw '{{"checkpoint": "{ck}", "width": 512, "max_width": 2048}}' \
            --out {res} | tail -12

## 5b. Method 5A on cubes (long)
Trained on cubes N = 3..7 (every cluster type; oblique centers need N ≥ 6), with the Q*-style move head.

In [ ]:
if RUN_5A_CUBE:
    out = f"{OUT}/m5a/cube.large.pt"
    if not os.path.exists(out):
        !python -m rubiks.solvers.m5a_tokens.train --puzzle cube --train-N 3-7 --val-N 8,10,14 \
            --steps {M5A_CUBE_STEPS} --move-head --device cuda --save-every 1000 --out {out} \
            2>&1 | tee -a {OUT}/logs/m5a_cube.log | grep -E "step|resumed|saved"

In [ ]:
if RUN_EVAL and RUN_5A_CUBE and os.path.exists(f"{OUT}/m5a/cube.large.pt"):
    res = f"{OUT}/results/m5a_cube.jsonl"
    if not os.path.exists(res):
        ck = f"{OUT}/m5a/cube.large.pt"
        !python -m rubiks.bench run --solver m5a_tokens --puzzle cube --N 3-8,10 --dataset random --limit 2 \
            --device cuda --time 1800 --kw '{{"checkpoint": "{ck}"}}' --out {res} | tail -10

## 6. Method 5B: per-cluster MLPs
One MLP per cluster type (corner, middle edge, wing, three center types), reused for every N. The repo's
small CPU-trained models are moved aside to `checkpoints/m5b/cpu_small/`; the new ones are written straight
to Drive (`checkpoints/m5b` becomes a link to `MyDrive/rubik_gpu/m5b`), so a rerun skips the clusters already
trained.

In [ ]:
import glob, os, shutil
if RUN_5B:
    drive_dir = f"{OUT}/m5b"
    if not os.path.islink("checkpoints/m5b"):
        os.makedirs("checkpoints/m5b_cpu_small", exist_ok=True)
        for f in glob.glob("checkpoints/m5b/*.pt") + glob.glob("checkpoints/m5b/*.json"):
            shutil.move(f, "checkpoints/m5b_cpu_small/")
        if not os.path.exists(f"{drive_dir}/tables"):
            shutil.copytree("checkpoints/m5b/tables", f"{drive_dir}/tables")
        shutil.rmtree("checkpoints/m5b")
        os.symlink(drive_dir, "checkpoints/m5b")
    !python -m rubiks.solvers.m5b_stages.train --puzzle cube --N 3-7 --steps {M5B_STEPS} --hidden {M5B_HIDDEN} \
        --device cuda 2>&1 | tee -a {OUT}/logs/m5b.log

In [ ]:
if RUN_EVAL and RUN_5B:
    res = f"{OUT}/results/m5b_cube_gpu.jsonl"
    if not os.path.exists(res):
        !python -m rubiks.bench run --solver m5b_stages --puzzle cube --N 3-8,10 --dataset random --limit 2 \
            --device cuda --time 1800 --out {res} | tail -10

## 7. Collect everything
Writes `MyDrive/rubik_gpu/rubik_gpu_outputs.zip` with every model, its `.json` training record (time, peak
GPU memory) and every result file.

**Back in the repo** (on your machine):
* `m4/cube<N>.large.pt`, `m4/square<N>.large.pt` → `checkpoints/m4/` (the solver prefers `.large.pt`)
* `m5a/square.large.pt`, `m5a/cube.large.pt` → pass as `--kw '{"checkpoint": "..."}'`, or copy to
  `checkpoints/m5a/square.pt` / `cube.pt` to make them the default
* `m5b/*.pt` + `*.json` → `checkpoints/m5b/` (replacing the CPU models; keep `tables/`)
* `results/*.jsonl` → `paper/results/` for the paper's tables

Files over ~5 MB stay out of git (`*.large.pt`); share them as GitHub release assets.

In [ ]:
import shutil
z = shutil.make_archive("/content/rubik_gpu_outputs", "zip", OUT)
shutil.copy(z, f"{OUT}/rubik_gpu_outputs.zip")
print("wrote", f"{OUT}/rubik_gpu_outputs.zip")
!ls -la {OUT}/m4 {OUT}/m5a {OUT}/results